In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/cicids2017").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data/cicids2017"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "docs/report/figures"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["BENIGN", "PORT_SCAN", "DOS", "BRUTE_FORCE"]
LABEL_MAP = {
    "BENIGN": "BENIGN",
    "PortScan": "PORT_SCAN",
    "DoS Hulk": "DOS",
    "DoS slowloris": "DOS",
    "DoS Slowhttptest": "DOS",
    "DoS GoldenEye": "DOS",
    "FTP-Patator": "BRUTE_FORCE",
    "SSH-Patator": "BRUTE_FORCE",
}
FEATURES = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Mean",
    "Flow IAT Mean",
    "SYN Flag Count",
    "ACK Flag Count",
    "RST Flag Count",
    "Average Packet Size",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Down/Up Ratio",
]

Matplotlib created a temporary cache directory at C:\Users\Tunc\AppData\Local\Temp\matplotlib-1xzylkvk because the default path (C:\Users\Tunc\.matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


## Dataset

Download CICIDS2017 `MachineLearningCSV.zip` from https://www.unb.ca/cic/datasets/ids-2017.html and extract it under `data/cicids2017/`.

The notebook expects CSV files such as `Friday-WorkingHours-Morning.pcap_ISCX.csv`, `Monday-WorkingHours.pcap_ISCX.csv`, and the other CICIDS2017 machine-learning CSVs directly inside that directory.

In [2]:
csv_paths = sorted(DATA_DIR.glob("*.csv"))
if not csv_paths:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR.resolve()}")

required_columns = set(FEATURES + ["Label"])
frames = []
for path in csv_paths:
    part = pd.read_csv(
        path,
        usecols=lambda column: column.strip() in required_columns,
        low_memory=False,
    )
    part.columns = part.columns.str.strip()
    frames.append(part)
    print(f"Loaded {path.name}: {len(part):,} rows")

df = pd.concat(frames, ignore_index=True)
print(f"Total loaded rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")

Loaded Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 rows


Loaded Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 rows


Loaded Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 rows


Loaded Monday-WorkingHours.pcap_ISCX.csv: 529,918 rows


Loaded Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602 rows


Loaded Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,366 rows


Loaded Tuesday-WorkingHours.pcap_ISCX.csv: 445,909 rows


Loaded Wednesday-workingHours.pcap_ISCX.csv: 692,703 rows
Total loaded rows: 2,830,743
Columns: ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Mean', 'Bwd Packet Length Mean', 'Flow IAT Mean', 'Fwd Packets/s', 'Bwd Packets/s', 'SYN Flag Count', 'RST Flag Count', 'ACK Flag Count', 'Down/Up Ratio', 'Average Packet Size', 'Label']


In [3]:
df["Label"] = df["Label"].astype(str).str.strip()
df = df[df["Label"].isin(LABEL_MAP)].copy()
df["target_label"] = df["Label"].map(LABEL_MAP)

class_counts = df["target_label"].value_counts().reindex(LABELS, fill_value=0)
print(class_counts)
missing_classes = [label for label in LABELS if class_counts[label] == 0]
if missing_classes:
    raise ValueError(f"Missing expected classes after filtering: {missing_classes}")

target_label
BENIGN         2273097
PORT_SCAN       158930
DOS             252661
BRUTE_FORCE      13835
Name: count, dtype: int64


In [4]:
benign = df[df["target_label"] == "BENIGN"]
other = df[df["target_label"] != "BENIGN"]
if len(benign) > 200_000:
    benign = benign.sample(n=200_000, random_state=42)

df_balanced = (
    pd.concat([benign, other], ignore_index=True)
    .sample(frac=1.0, random_state=42)
    .reset_index(drop=True)
)
print(df_balanced["target_label"].value_counts().reindex(LABELS, fill_value=0))

target_label
BENIGN         200000
PORT_SCAN      158930
DOS            252661
BRUTE_FORCE     13835
Name: count, dtype: int64


In [5]:
missing_features = [feature for feature in FEATURES if feature not in df_balanced.columns]
if missing_features:
    raise KeyError(f"Missing CICIDS feature columns: {missing_features}")

X = df_balanced[FEATURES].copy()
y_labels = df_balanced["target_label"].copy()
print(f"Feature matrix before cleaning: {X.shape}")

Feature matrix before cleaning: (625426, 15)


In [6]:
for feature in FEATURES:
    X[feature] = pd.to_numeric(X[feature], errors="coerce")

X = X.replace([np.inf, -np.inf], np.nan)
clean_mask = X.notna().all(axis=1)
dropped_rows = int((~clean_mask).sum())
X = X.loc[clean_mask].reset_index(drop=True)
y_labels = y_labels.loc[clean_mask].reset_index(drop=True)
print(f"Dropped rows with NaN/inf: {dropped_rows:,}")
print(f"Feature matrix after cleaning: {X.shape}")

Dropped rows with NaN/inf: 0


Feature matrix after cleaning: (625426, 15)


In [7]:
label_to_id = {label: index for index, label in enumerate(LABELS)}
y = y_labels.map(label_to_id).astype(int).to_numpy()
X_values = X.to_numpy(dtype=np.float64)
print(label_to_id)

{'BENIGN': 0, 'PORT_SCAN': 1, 'DOS': 2, 'BRUTE_FORCE': 3}


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X_values,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)
print(f"Train rows: {len(y_train):,}; test rows: {len(y_test):,}")

Train rows: 500,340; test rows: 125,086


In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Scaled train/test feature matrices.")

Scaled train/test feature matrices.


In [10]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
)
model.fit(X_train_scaled, y_train)
print("RandomForestClassifier trained.")

RandomForestClassifier trained.


In [11]:
y_pred = model.predict(X_test_scaled)
report = classification_report(
    y_test,
    y_pred,
    target_names=LABELS,
    output_dict=True,
    zero_division=0,
)
cm = confusion_matrix(y_test, y_pred, labels=list(range(len(LABELS))))
macro_f1 = float(report["macro avg"]["f1-score"])

metrics = {
    "labels": LABELS,
    "features": FEATURES,
    "macro_f1": macro_f1,
    "per_class": {
        label: {
            "precision": float(report[label]["precision"]),
            "recall": float(report[label]["recall"]),
            "f1-score": float(report[label]["f1-score"]),
            "support": int(report[label]["support"]),
        }
        for label in LABELS
    },
    "confusion_matrix": cm.tolist(),
    "classification_report": report,
}

with (MODELS_DIR / "metrics.json").open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

brute_force_f1 = metrics["per_class"]["BRUTE_FORCE"]["f1-score"]
print(f"macro_f1 = {macro_f1:.6f}")
print(f"BRUTE_FORCE f1 = {brute_force_f1:.6f}")
if macro_f1 < 0.93:
    raise RuntimeError(f"Acceptance failed: macro F1 {macro_f1:.4f} < 0.93")
if brute_force_f1 < 0.75:
    raise RuntimeError(f"Acceptance failed: BRUTE_FORCE F1 {brute_force_f1:.4f} < 0.75")

macro_f1 = 0.938958
BRUTE_FORCE f1 = 0.796855


In [12]:
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("CICIDS2017 Random Forest Confusion Matrix")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print(FIGURES_DIR / "confusion_matrix.png")

C:\Users\Tunc\Desktop\introProjesi\docs\report\figures\confusion_matrix.png


In [13]:
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
plt.figure(figsize=(9, 6))
sns.barplot(x=importance.values, y=importance.index, color="#4C78A8")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 15 Random Forest Feature Importances")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print(FIGURES_DIR / "feature_importance.png")

C:\Users\Tunc\Desktop\introProjesi\docs\report\figures\feature_importance.png


In [14]:
joblib.dump(model, MODELS_DIR / "rf_model.pkl")
joblib.dump(scaler, MODELS_DIR / "scaler.pkl")

summary = pd.DataFrame(
    [
        {
            "macro_f1": macro_f1,
            "train_rows": len(y_train),
            "test_rows": len(y_test),
            "features": len(FEATURES),
            "model": type(model).__name__,
        }
    ]
)
print("Training complete. Artifacts saved under models/ and docs/report/figures/.")
display(summary)

Training complete. Artifacts saved under models/ and docs/report/figures/.


,macro_f1,train_rows,test_rows,features,model
0,0.938958,500340,125086,15,RandomForestClassifier
